In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
import tensorflow as tf
from tensorflow import kerasv
from tensorflow.keras import layers, regularizers
from tensorflow.keras import backend as K
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from sklearn.utils.class_weight import compute_class_weight
df = pd.read_csv("model_data.csv")

cols       = df.columns.tolist()
target_col = cols[0]
dummy_cols = cols[1:4]
cat_col    = cols[5]
quant_cols = cols[6:]

# Konvertér binære kolonner — fyld NaN med mode før int-cast
for col in dummy_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")
    df[col] = df[col].fillna(df[col].mode()[0])
    df[col] = df[col].astype(int)

y = df[target_col].astype(int).values    # ← astype(int) tilføjet

# One-hot encoding af kategorisk variabel — drop_first=False bevarer ALLE kategorier
dom_dummies = pd.get_dummies(
    df[cat_col].astype(str), prefix=cat_col, drop_first=False
)

# Saml: dummy-kolonner | one-hot-kolonner | kvantitative kolonner
X_df = pd.concat([df[dummy_cols], dom_dummies, df[quant_cols]], axis=1)

# ── 4. Opdel i trænings- og testsæt (80/20) FØR skalering ──────────────────
X_train_df, X_test_df, y_train, y_test = train_test_split(
    X_df, y, test_size=0.20, random_state=1245, stratify=y
)

scaler      = StandardScaler()
X_train_arr = X_train_df.copy()
X_test_arr  = X_test_df.copy()

X_train_arr[quant_cols] = scaler.fit_transform(X_train_df[quant_cols])  # fit + transform
X_test_arr[quant_cols]  = scaler.transform(X_test_df[quant_cols])       # kun transform

# Træningsdata: Her beregnes gennemsnit og standardafvigelse, og data skaleres (fit_transform).
# Testdata: Her bruges de allerede beregnede værdier fra træningsdata til at skalere testdata (transform).
# Dette forhindrer data leakage og sikrer, at modellen kun "ser" træningsdata under preprocessing.

# Konvertér til numpy float32 arrays (Keras-format)
X_train = X_train_arr.values.astype(np.float32)
X_test  = X_test_arr.values.astype(np.float32)

resultater = {}

ImportError: cannot import name 'kerasv' from 'tensorflow' (/usr/local/lib/python3.12/dist-packages/tensorflow/__init__.py)

In [ ]:

# =============================================================================
# MODEL 01 — Simpel baseline
# =============================================================================
# Den enklest mulige arkitektur: to skjulte lag uden regularisering.
# Bruges som baseline — alle efterfølgende modeller bør slå denne.
# Ingen dropout eller batch normalization — risikerer overfitting.
K.clear_session()
tf.random.set_seed(42)

model_01 = keras.Sequential([
    layers.Input(shape=(X_train.shape[1],)),    # Inputlag: antal features
    layers.Dense(64, activation="relu"),        # Skjult lag 1: 64 neuroner, ReLU aktivering
    layers.Dense(32, activation="relu"),        # Skjult lag 2: 32 neuroner, ReLU aktivering
    layers.Dense(1,  activation="sigmoid")      # Outputlag: sigmoid giver P(churnet) ∈ [0,1]
], name="model_01")

# binary_crossentropy er standardtabsfunktionen til binær klassifikation.
# Adam er en adaptiv optimizer der justerer learning rate per parameter.
model_01.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.AUC(name="auc"),
             tf.keras.metrics.Precision(), tf.keras.metrics.Recall()]
)
model_01.summary()

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)
class_weight_dict = dict(enumerate(class_weights))

history_01 = model_01.fit(
    X_train, y_train,
    class_weight=class_weight_dict,
    epochs=50,
    batch_size=64,                  # 64 observationer per gradient-opdatering
    validation_split=0.20,          # 20% af træningsdata til løbende validering

    verbose=1
)

loss, acc, auc, precision, recall = model_01.evaluate(X_test, y_test, verbose=0)
resultater["Model 01"] = {"loss": loss, "accuracy": acc, "auc": auc,
                           "precision": precision, "recall": recall}
print(f"\nModel 01 → Loss: {loss:.4f} | Acc: {acc:.4f} | AUC: {auc:.4f} | Precision: {precision:.4f} | Recall: {recall:.4f}")


Model: "model_01"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │         1,152 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,265 (12.75 KB)

 Trainable params: 3,265 (12.75 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
13/13 ━━━━━━━━━━━━━━━━━━━━ 4s 78ms/step - accuracy: 0.7555 - auc: 0.6335 - loss: 0.6595 - precision: 0.8571 - recall: 0.8471 - val_accuracy: 0.7073 - val_auc: 0.7168 - val_loss: 0.6254 - val_precision: 0.8531 - val_recall: 0.7578
Epoch 2/50
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.7347 - auc: 0.7441 - loss: 0.6223 - precision: 0.8944 - recall: 0.7721 - val_accuracy: 0.7220 - val_auc: 0.7263 - val_loss: 0.6207 - val_precision: 0.8939 - val_recall: 0.7329
Epoch 3/50
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.7298 - auc: 0.7727 - loss: 0.5978 - precision: 0.9091 - recall: 0.7500 - val_accuracy: 0.7220 - val_auc: 0.7405 - val_loss: 0.6043 - val_precision: 0.9000 - val_recall: 0.7267
Epoch 4/50
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.7445 - auc: 0.7935 - loss: 0.5755 - precision: 0.9124 - recall: 0.7662 - val_accuracy: 0.7268 - val_auc: 0.7571 - val_loss: 0.5828 - val_precision: 0.9008 - val_recall: 0.7329
Epoch 5/50
13/13 ━━━━━━━━━━━━━━━━━━━

In [ ]:
# =============================================================================
# MODEL 02 — Dropout regularisering
# =============================================================================
# Tilføjer Dropout til baseline-arkitekturen. Dropout slukker tilfældigt en
# andel af neuroner under træning, hvilket tvinger netværket til at lære mere
# robuste representationer og reducerer overfitting.
# Dropout(0.3) = 30% af neuroner deaktiveres i hvert træningsskridt.
K.clear_session()
tf.random.set_seed(42)

model_02 = keras.Sequential([
    layers.Input(shape=(X_train.shape[1],)),
    layers.Dense(128, activation="relu"),       # Større første lag end model_01
    layers.Dropout(0.3),                        # 30% dropout efter lag 1
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.2),                        # 20% dropout efter lag 2
    layers.Dense(1, activation="sigmoid")
], name="model_02")

model_02.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.AUC(name="auc"),
             tf.keras.metrics.Precision(), tf.keras.metrics.Recall()]
)
model_02.summary()

# EarlyStopping: stopper træning hvis val_loss ikke forbedres i 5 epochs
# restore_best_weights=True ruller tilbage til den bedste epoch ved stop
early_stopping_02 = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)

history_02 = model_02.fit(
    X_train, y_train,
    epochs=100,                         # EarlyStopping stopper før hvis nødvendigt
    batch_size=64,
    validation_split=0.20,
    callbacks=[early_stopping_02],
    verbose=1
)

loss, acc, auc, precision, recall = model_02.evaluate(X_test, y_test, verbose=0)
resultater["Model 02"] = {"loss": loss, "accuracy": acc, "auc": auc,
                           "precision": precision, "recall": recall}
print(f"\nModel 02 → Loss: {loss:.4f} | Acc: {acc:.4f} | AUC: {auc:.4f} | Precision: {precision:.4f} | Recall: {recall:.4f}")


Model: "model_02"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │         2,304 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 10,625 (41.50 KB)

 Trainable params: 10,625 (41.50 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 6s 91ms/step - accuracy: 0.7127 - auc: 0.5573 - loss: 0.5828 - precision: 0.8326 - recall: 0.8191 - val_accuracy: 0.7854 - val_auc: 0.6979 - val_loss: 0.4988 - val_precision: 0.7854 - val_recall: 1.0000
Epoch 2/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.8313 - auc: 0.6500 - loss: 0.4433 - precision: 0.8313 - recall: 1.0000 - val_accuracy: 0.7854 - val_auc: 0.7277 - val_loss: 0.4923 - val_precision: 0.7854 - val_recall: 1.0000
Epoch 3/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.8313 - auc: 0.7115 - loss: 0.4166 - precision: 0.8313 - recall: 1.0000 - val_accuracy: 0.7854 - val_auc: 0.7445 - val_loss: 0.4822 - val_precision: 0.7854 - val_recall: 1.0000
Epoch 4/100
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.8313 - auc: 0.7556 - loss: 0.3969 - precision: 0.8313 - recall: 1.0000 - val_accuracy: 0.7854 - val_auc: 0.7556 - val_loss: 0.4637 - val_precision: 0.7854 - val_recall: 1.0000
Epoch 5/100
13/13 ━━━━━━━━━━━━━━

In [ ]:
# =============================================================================
# MODEL 03 — Dybere netværk + blødgjorte class weights + t=0.40
# =============================================================================
# Tilføjer et ekstra skjult lag ift. model_02 og bruger blødgjorte class weights
# ({0:2.0, 1:0.8}) frem for "balanced". Dette løfter Recall markant fordi
# modellen ikke overkompenserer for majoritetsklassen.
# Threshold sættes til 0.40 (frem for standard 0.50) da threshold-analysen
# viste at t=0.40 giver den bedste F1-score og højeste accuracy.
K.clear_session()
tf.random.set_seed(42)

model_03 = keras.Sequential([
    layers.Input(shape=(X_train.shape[1],)),
    layers.Dense(64, activation="relu"),        # Lag 1
    layers.Dropout(0.3),
    layers.Dense(64, activation="relu"),        # Lag 2 — samme størrelse som lag 1
    layers.Dropout(0.3),
    layers.Dense(32, activation="relu"),        # Lag 3 — reducerer til 32 neuroner
    layers.Dropout(0.2),
    layers.Dense(1, activation="sigmoid")
], name="model_03")

model_03.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.AUC(name="auc"),
             tf.keras.metrics.Precision(), tf.keras.metrics.Recall()]
)
model_03.summary()

# EarlyStopping: stopper træning hvis val_loss ikke forbedres i 5 epochs
# restore_best_weights=True ruller tilbage til den bedste epoch ved stop
early_stopping_03 = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)

history_03 = model_03.fit(
    X_train, y_train,
    epochs=150,                         # EarlyStopping stopper før hvis nødvendigt
    batch_size=64,
    validation_split=0.20,
    callbacks=[early_stopping_03],
    verbose=1
)

loss, acc, auc, precision, recall = model_03.evaluate(X_test, y_test, verbose=0)
resultater["Model 03"] = {"loss": loss, "accuracy": acc, "auc": auc,
                           "precision": precision, "recall": recall}
print(f"\nModel 03 → Loss: {loss:.4f} | Acc: {acc:.4f} | AUC: {auc:.4f} | Precision: {precision:.4f} | Recall: {recall:.4f}")


Model: "model_03"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │         1,152 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,425 (29.00 KB)

 Trainable params: 7,425 (29.00 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/150
13/13 ━━━━━━━━━━━━━━━━━━━━ 3s 45ms/step - accuracy: 0.7164 - auc: 0.4902 - loss: 0.6327 - precision: 0.8394 - recall: 0.8147 - val_accuracy: 0.7854 - val_auc: 0.6177 - val_loss: 0.5417 - val_precision: 0.7854 - val_recall: 1.0000
Epoch 2/150
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.8301 - auc: 0.5042 - loss: 0.5084 - precision: 0.8311 - recall: 0.9985 - val_accuracy: 0.7854 - val_auc: 0.6632 - val_loss: 0.4984 - val_precision: 0.7854 - val_recall: 1.0000
Epoch 3/150
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.8313 - auc: 0.6096 - loss: 0.4512 - precision: 0.8313 - recall: 1.0000 - val_accuracy: 0.7854 - val_auc: 0.7052 - val_loss: 0.4908 - val_precision: 0.7854 - val_recall: 1.0000
Epoch 4/150
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.8313 - auc: 0.6660 - loss: 0.4296 - precision: 0.8313 - recall: 1.0000 - val_accuracy: 0.7854 - val_auc: 0.7237 - val_loss: 0.4803 - val_precision: 0.7854 - val_recall: 1.0000
Epoch 5/150
13/13 ━━━━━━━━━━━━━━

In [ ]:
# =============================================================================
# MODEL 04 — Batch Normalization
# =============================================================================
# Erstatter ren dropout med Batch Normalization (BN). BN normaliserer
# aktiveringsmønstret inden for hvert lag per batch, hvilket stabiliserer
# træningen og kan tillade højere learning rates. BN placeres MELLEM Dense-laget
# og aktiveringsfunktionen for bedste effekt. Batch size 32 (frem for 64) giver
# BN bedre estimater af mean og variance per batch.
K.clear_session()
tf.random.set_seed(42)

model_04 = keras.Sequential([
    layers.Input(shape=(X_train.shape[1],)),
    layers.Dense(128, activation=None),         # Ingen aktivering — BN kommer først
    layers.BatchNormalization(),                # Normaliserer inden aktivering
    layers.Activation("relu"),
    layers.Dropout(0.2),
    layers.Dense(64, activation=None),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Dropout(0.3),
    layers.Dense(1, activation="sigmoid")
], name="model_04")

model_04.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.AUC(name="auc"),
             tf.keras.metrics.Precision(), tf.keras.metrics.Recall()]
)
model_04.summary()

# EarlyStopping: stopper træning hvis val_loss ikke forbedres i 5 epochs
# restore_best_weights=True ruller tilbage til den bedste epoch ved stop
early_stopping_04 = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)

history_04 = model_04.fit(
    X_train, y_train,
    epochs=200,                         # EarlyStopping stopper før hvis nødvendigt
    batch_size=32,                      # Mindre batch — BN fungerer bedre med mere variation
    validation_split=0.20,
    callbacks=[early_stopping_04],
    verbose=1
)

loss, acc, auc, precision, recall = model_04.evaluate(X_test, y_test, verbose=0)
resultater["Model 04"] = {"loss": loss, "accuracy": acc, "auc": auc,
                           "precision": precision, "recall": recall}
print(f"\nModel 04 → Loss: {loss:.4f} | Acc: {acc:.4f} | AUC: {auc:.4f} | Precision: {precision:.4f} | Recall: {recall:.4f}")

Model: "model_04"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │         2,304 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,393 (44.50 KB)

 Trainable params: 11,009 (43.00 KB)

 Non-trainable params: 384 (1.50 KB)

Epoch 1/200
26/26 ━━━━━━━━━━━━━━━━━━━━ 6s 25ms/step - accuracy: 0.4450 - auc: 0.6364 - loss: 0.8593 - precision: 0.8951 - recall: 0.3765 - val_accuracy: 0.7951 - val_auc: 0.8065 - val_loss: 0.6261 - val_precision: 0.8790 - val_recall: 0.8571
Epoch 2/200
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7103 - auc: 0.7210 - loss: 0.5919 - precision: 0.9094 - recall: 0.7235 - val_accuracy: 0.8146 - val_auc: 0.8414 - val_loss: 0.5529 - val_precision: 0.8220 - val_recall: 0.9752
Epoch 3/200
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8142 - auc: 0.7275 - loss: 0.4721 - precision: 0.8894 - recall: 0.8868 - val_accuracy: 0.8049 - val_auc: 0.8404 - val_loss: 0.5005 - val_precision: 0.8040 - val_recall: 0.9938
Epoch 4/200
26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8203 - auc: 0.7829 - loss: 0.4036 - precision: 0.8666 - recall: 0.9265 - val_accuracy: 0.8098 - val_auc: 0.8438 - val_loss: 0.4624 - val_precision: 0.8050 - val_recall: 1.0000
Epoch 5/200
26/26 ━━━━━━━━━━━━━━━━━

In [ ]:
# MODEL 05 — L2 regularisering + gradient clipping + callbacks
# =============================================================================
# Den mest regulariserede arkitektur. L2 (weight decay) straffer store vægte
# i tabsfunktionen, hvilket tvinger modellen til enklere løsninger.
# Gradient clipping (clipvalue=1.0) forhindrer eksploderende gradienter i
# dybe netværk. EarlyStopping stopper træning automatisk når val_loss holder
# op med at falde — epochs=100 er derfor et maksimum, ikke et mål.
# ReduceLROnPlateau halverer learning rate når val_loss stagnerer.
K.clear_session()
tf.random.set_seed(42)

model_05 = Sequential([
    layers.Input(shape=(X_train.shape[1],)),
    layers.Dense(32, activation=None, kernel_regularizer=regularizers.l2(0.09)),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Dropout(0.3),
    layers.Dense(16, activation=None, kernel_regularizer=regularizers.l2(0.09)),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Dropout(0.3),
    layers.Dense(16, activation=None, kernel_regularizer=regularizers.l2(0.05)),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Dropout(0.3),
    layers.Dense(16, activation=None, kernel_regularizer=regularizers.l2(0.04)),
    layers.BatchNormalization(),
    layers.Activation("relu"),
    layers.Dropout(0.3),
    layers.Dense(1, activation="sigmoid")
], name="model_05")

# Adam med gradient clipping — sikrer stabile opdateringer i det dybe netværk
optimizer_05 = Adam(learning_rate=0.001, clipvalue=1.0)

model_05.compile(
    optimizer=optimizer_05,
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.AUC(name="auc"),
             tf.keras.metrics.Precision(), tf.keras.metrics.Recall()]
)
model_05.summary()

# EarlyStopping: stopper træning hvis val_loss ikke forbedres i 5 epochs
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True       # Ruller tilbage til bedste epoch ved stop
)

# ReduceLROnPlateau: halverer learning rate hvis val_loss stagnerer i 5 epochs
lr_scheduler = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=5,
    min_lr=1e-6                     # Sænker aldrig learning rate under 0.000001
)

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)

history_05 = model_05.fit(
    X_train, y_train,
    epochs=200,                     # EarlyStopping stopper før hvis nødvendigt
    batch_size=64,
    validation_split=0.20,
    callbacks=[early_stopping, lr_scheduler],
    verbose=1
)

loss, acc, auc, precision, recall = model_05.evaluate(X_test, y_test, verbose=0)
resultater["Model 05"] = {"loss": loss, "accuracy": acc, "auc": auc,
                           "precision": precision, "recall": recall}
print(f"\nModel 05 → Loss: {loss:.4f} | Acc: {acc:.4f} | AUC: {auc:.4f} | Precision: {precision:.4f} | Recall: {recall:.4f}")


Model: "model_05"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 32)             │           576 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 32)             │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 16)             │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 16)             │           272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 16)             │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 16)             │           272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 16)             │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_3 (Activation)       │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,985 (7.75 KB)

 Trainable params: 1,825 (7.13 KB)

 Non-trainable params: 160 (640.00 B)

Epoch 1/200
13/13 ━━━━━━━━━━━━━━━━━━━━ 7s 58ms/step - accuracy: 0.7078 - auc: 0.5335 - loss: 5.8076 - precision: 0.8356 - recall: 0.8074 - val_accuracy: 0.7756 - val_auc: 0.3820 - val_loss: 5.5877 - val_precision: 0.7833 - val_recall: 0.9876 - learning_rate: 0.0010
Epoch 2/200
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.7335 - auc: 0.5188 - loss: 5.3406 - precision: 0.8348 - recall: 0.8471 - val_accuracy: 0.7805 - val_auc: 0.3922 - val_loss: 5.1432 - val_precision: 0.7843 - val_recall: 0.9938 - learning_rate: 0.0010
Epoch 3/200
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.7738 - auc: 0.5518 - loss: 4.8803 - precision: 0.8551 - recall: 0.8765 - val_accuracy: 0.7805 - val_auc: 0.4099 - val_loss: 4.7277 - val_precision: 0.7843 - val_recall: 0.9938 - learning_rate: 0.0010
Epoch 4/200
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.7482 - auc: 0.4745 - loss: 4.5049 - precision: 0.8292 - recall: 0.8779 - val_accuracy: 0.7805 - val_auc: 0.4406 - val_loss: 4.3413 - val

In [ ]:
# SAMMENLIGNING AF ALLE 5 MODELLER
# =============================================================================
# F1-score er harmonisk gennemsnit af Precision og Recall — en god samlet metric
# når datasættet er skævt. Vi beregner den for alle modeller for ensartet sammenligning.
# For model_03 bruger vi allerede de t=0.40 tal — de andre bruger standard t=0.50.

print("\n" + "="*85)
print("SAMMENLIGNING AF ALLE MODELLER")
print("="*85)
print(f"{'Model':<22} {'Loss':>7} {'Accuracy':>10} {'AUC':>7} {'Precision':>11} {'Recall':>8} {'F1':>7}")
print("-"*85)

for navn, r in resultater.items():
    # Beregn F1 hvis ikke allerede gemt (model_03 har den, resten beregnes her)
    if "f1" not in r:
        r["f1"] = 2 * (r["precision"] * r["recall"]) / (r["precision"] + r["recall"])
    print(f"{navn:<22} {r['loss']:>7.4f} {r['accuracy']:>10.4f} {r['auc']:>7.4f} "
          f"{r['precision']:>11.4f} {r['recall']:>8.4f} {r['f1']:>7.4f}")

print("="*85)
print("""
FORTOLKNING:
  Loss       → Lavere er bedre. Måler hvor sikker modellen er i sine forudsigelser.
  Accuracy   → Andel korrekte forudsigelser. Kan være vildledende med 82% churn-rate.
  AUC        → Evnen til at skelne churnere fra ikke-churnere uafhængigt af threshold.
               0.90 = meget god, 0.80 = acceptabel, under 0.70 = svag.
  Precision  → Af dem vi kalder "churner": hvor mange churner faktisk?
               Høj Precision = få falske alarmer.
  Recall     → Af dem der faktisk churner: hvor mange fanger vi?
               Høj Recall = få churnere slipper igennem uopdaget. VIGTIGST for JP.
  F1         → Harmonisk gennemsnit af Precision og Recall. God samlet metric.

""")


SAMMENLIGNING AF ALLE MODELLER
Model                     Loss   Accuracy     AUC   Precision   Recall      F1
-------------------------------------------------------------------------------------
Model 01                0.3664     0.8398  0.8803      0.9126   0.8910  0.9017
Model 02                0.3275     0.8477  0.8852      0.8707   0.9573  0.9120
Model 03                0.2997     0.8477  0.8959      0.8739   0.9526  0.9116
Model 04                0.3418     0.8516  0.8557      0.8681   0.9668  0.9148
Model 05                0.4430     0.8242  0.8816      0.8242   1.0000  0.9036

FORTOLKNING:
  Loss       → Lavere er bedre. Måler hvor sikker modellen er i sine forudsigelser.
  Accuracy   → Andel korrekte forudsigelser. Kan være vildledende med 82% churn-rate.
  AUC        → Evnen til at skelne churnere fra ikke-churnere uafhængigt af threshold.
               0.90 = meget god, 0.80 = acceptabel, under 0.70 = svag.
  Precision  → Af dem vi kalder "churner": hvor mange churner fakt